# Running baseline models
Normal classification of diagnoses

## Setup

In [ ]:
from pathlib import Path
import subprocess
import time
import warnings
import numpy as np
import pandas as pd
import sklearn
import mlflow
import dagshub
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, log_loss, roc_auc_score)
from uuid import uuid4


TRAIN_PATH = Path("C:/Users/Hzaab/SDP-Sequential-Diagnosis/afabench/Afabench_Source/afabench/extra/data/misc/ddxplus_full.csv")
TEST_PATH = Path("C:/Users/Hzaab/SDP-Sequential-Diagnosis/afabench/Afabench_Source/afabench/extra/data/misc/ddxplus_test.csv")
TARGET = "pathology"
version = "Fifth"
fraction = 0.05
seed = 42
val_size = 0.2
owner = "HindAlz"
repo = "Sequential-Diagnosis"
experiment = "sdp-sequential-diagnosis"


c:\Users\Hzaab\SDP-Sequential-Diagnosis\.venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
dagshub.init(repo_owner=owner, repo_name=repo, mlflow=True)
mlflow.set_experiment(experiment)
commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True
).strip()

print("Commit:", commit)

Accessing as HindAlz

Initialized MLflow to track repo "HindAlz/Sequential-Diagnosis"

Repository HindAlz/Sequential-Diagnosis initialized!

Commit: 38c2100cd9aa2390de0f120ed85b45af9cec8ef2


## Preprocessing

In [ ]:
train_full = pd.read_csv(TRAIN_PATH)
test_full = pd.read_csv(TEST_PATH)

feature_columns = [c for c in train_full.columns if c != TARGET]
numeric_columns = train_full[feature_columns].select_dtypes(include=np.number).columns.tolist()
categorical_columns = train_full[feature_columns].select_dtypes(exclude=np.number).columns.tolist()

def sample_rows(df):
    if fraction == 1:
        return df.copy()

    selected, _ = train_test_split(df, train_size=fraction, stratify=df[TARGET], random_state=seed)
    
    return selected.copy()

train = sample_rows(train_full)
test = sample_rows(test_full)

X = train[feature_columns]
y = train[TARGET]

X_test = test[feature_columns]
y_test = test[TARGET]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=val_size, stratify=y, random_state=seed)

print("Full rows: train= ",len(train_full),":, test= ", len(test_full),":")
print("Sampled rows: train= ", len(train),":, test= ", len(test),":")
print("Fit= ", len(X_train),", validation= ", len(X_val), " classes=", y.nunique())

del train_full, test_full


Missing values:
Train:
Series([], dtype: int64)

Test:
Series([], dtype: int64)

Infinite values:
Train:
E_0          0
E_1          0
E_10         0
E_100        0
E_101        0
            ..
E_99         0
age          0
pain         0
sex          0
pathology    0
Length: 810, dtype: int64

Test:
E_0          0
E_1          0
E_10         0
E_100        0
E_101        0
            ..
E_99         0
age          0
pain         0
sex          0
pathology    0
Length: 810, dtype: int64

Full rows: train=1,025,602, test=134,529
Sampled rows: train=51,280, test=6,726
Fit=41,024; validation=10,256; classes=49


In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(y_train)

y_train = label_encoder.transform(y_train)
y_val = label_encoder.transform(y_val)
y_test = label_encoder.transform(y_test)

y = label_encoder.transform(y)

## models general settings

In [ ]:
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("scale", StandardScaler()),
    ]), numeric_columns),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING", keep_empty_features=True)),
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ]), categorical_columns),
], sparse_threshold=1.0)

models = {
    "RF": RandomForestClassifier(n_estimators=100, max_depth=20, min_samples_leaf=2, n_jobs=-1, random_state=seed),
    "DT": DecisionTreeClassifier(max_depth=20, min_samples_leaf=2, random_state=seed),
    "MLP": MLPClassifier(hidden_layer_sizes=(64,), max_iter=100, batch_size="auto", random_state=seed),
    "LOGREG": LogisticRegression(max_iter=500, random_state=seed),
    "ET": ExtraTreesClassifier(n_estimators=100, max_depth=20, min_samples_leaf=2, n_jobs=-1, random_state=seed),
    "DUMMY": DummyClassifier(strategy="prior", random_state=seed),
    "XGB": XGBClassifier(device="cuda", tree_method="hist", n_estimators=100, max_depth=6, learning_rate=0.1, subsample=0.8,
        colsample_bytree=0.8, max_bin=128, random_state=seed, n_jobs=2)
}


In [ ]:
def evaluate(model, features, target):
    predictions = model.predict(features)
    probabilities = model.predict_proba(features)
    classes = model.classes_

    metrics = {
        "accuracy": accuracy_score(target, predictions),
        "precision": precision_score(target, predictions, labels=classes, average="macro", zero_division=0),
        "recall": recall_score(target, predictions, labels=classes, average="macro", zero_division=0),
        "f1": f1_score(target, predictions, labels=classes, average="macro", zero_division=0),
        "loss": log_loss(target, probabilities, labels=classes),
    }
    
    if set(target) == set(classes):
        metrics["AUC"] = (
            roc_auc_score(target == classes[1], probabilities[:, 1])
            if len(classes) == 2 else
            roc_auc_score(target, probabilities, labels=classes,  multi_class="ovr", average="macro")
        )
    return {key: float(value) for key, value in metrics.items()}


In [ ]:
results = []
trial_configs = {}

In [ ]:

def run_trial(model_type, trial_name="trial", **hyperparams):
    estimator = clone(models[model_type]).set_params(**hyperparams)
    actual_params = estimator.get_params(deep=False)

    model = Pipeline([
        ("preprocess", clone(preprocessor)),
        ("classifier", estimator),
    ])

    trial_slug = "_".join(trial_name.strip().lower().split())
    trial_id = uuid4().hex[:8]

    run_name = (
        f"{version}_{model_type}_"
        f"{trial_slug}_validation_seed{actual_params.get('random_state', seed)}"
        f"_{trial_id}"
    )

    with mlflow.start_run(run_name=run_name):
        mlflow.set_tags({
            "mlflow.source.git.commit": commit,
            "commit": str(commit),
            "dataset_version": version,
            "split": "validation",
            "trial_name": trial_name,
            "metric_average": "macro",
            "loss_definition": "log_loss",
        })

        mlflow.log_params({
            "model": model_type,
            "seed": actual_params.get("random_state", seed),
            "sample_fraction": fraction,
            "validation_size": val_size,
            "target": TARGET,
            "n_features_raw": len(feature_columns),
            "n_fit": len(X_train),
            "n_eval": len(X_val),
            "sklearn_version": sklearn.__version__,
            **actual_params,  # Logs the parameters actually used.
        })

        started = time.perf_counter()
        model.fit(X_train, y_train)
        fit_seconds = time.perf_counter() - started

        metrics = evaluate(model, X_val, y_val)
        mlflow.log_metrics({
            **metrics,
            "fit_seconds": fit_seconds,
        })

        if "AUC" not in metrics:
            mlflow.set_tag(
                "auc_note",
                "Skipped: validation does not contain every training class",
            )

        run_id = mlflow.active_run().info.run_id

    row = {
        "model": model_type,
        "trial": trial_name,
        **metrics,
        "fit_seconds": fit_seconds,
        "hyperparams": hyperparams.copy(),
        "run_id": run_id,
    }

    results.append(row)

    # Save exact settings for refitting the winning configuration later.
    trial_configs[run_id] = {
        "model_type": model_type,
        "params": actual_params.copy(),
    }

    print("Run:", run_name)
    print("Run ID:", run_id)
    display(pd.DataFrame([row]))

    return model

## models

In [ ]:
xgb_model= run_trial("XGB",
    trial_name="gpu-depth6-100trees", n_estimators=100, max_depth=6, learning_rate=0.1)

c:\Users\Hzaab\SDP-Sequential-Diagnosis\.venv312\Lib\site-packages\xgboost\core.py:569: UserWarning: [16:49:34] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


🏃 View run hind_Fifth_XGB_gpu-depth6-100trees_validation_seed42_35753794 at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0/runs/bd0697f7dffc4ba2964d29080f5c994a
🧪 View experiment at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0
Run: hind_Fifth_XGB_gpu-depth6-100trees_validation_seed42_35753794
Run ID: bd0697f7dffc4ba2964d29080f5c994a


,model,trial,accuracy,precision,recall,f1,loss,AUC,fit_seconds,hyperparams,run_id
0,XGB,gpu-depth6-100trees,0.996977,0.997163,0.99637,0.996662,0.012998,0.99996,19.952889,"{'n_estimators': 100, 'max_depth': 6, 'learnin...",bd0697f7dffc4ba2964d29080f5c994a


In [ ]:
rf_model= run_trial("RF",
    trial_name="200trees-depth15", n_estimators=200, max_depth=15, min_samples_leaf=2)

🏃 View run hind_Fifth_RF_200trees-depth15_validation_seed42_96ed9c5a at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0/runs/ea874eb639174ac087d4ee650f1b0064
🧪 View experiment at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0
Run: hind_Fifth_RF_200trees-depth15_validation_seed42_96ed9c5a
Run ID: ea874eb639174ac087d4ee650f1b0064


,model,trial,accuracy,precision,recall,f1,loss,AUC,fit_seconds,hyperparams,run_id
0,RF,200trees-depth15,0.986154,0.989226,0.957521,0.965452,0.776168,0.999826,3.408822,"{'n_estimators': 200, 'max_depth': 15, 'min_sa...",ea874eb639174ac087d4ee650f1b0064


In [ ]:
dt_model = run_trial("DT",
    trial_name="depth10", max_depth=10, min_samples_leaf=5)

🏃 View run hind_Fifth_DT_depth10_validation_seed42_e2c43bb0 at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0/runs/0afe3f1c2422472c850c5a39ba519ef4
🧪 View experiment at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0
Run: hind_Fifth_DT_depth10_validation_seed42_e2c43bb0
Run ID: 0afe3f1c2422472c850c5a39ba519ef4


,model,trial,accuracy,precision,recall,f1,loss,AUC,fit_seconds,hyperparams,run_id
0,DT,depth10,0.421509,0.34751,0.311847,0.308567,2.166304,0.787972,3.199208,"{'max_depth': 10, 'min_samples_leaf': 5}",0afe3f1c2422472c850c5a39ba519ef4


In [ ]:
mlp_model = run_trial("MLP",
    trial_name="128-64", hidden_layer_sizes=(128, 64), learning_rate_init=0.001, alpha=0.001, max_iter=200)

🏃 View run hind_Fifth_MLP_128-64_validation_seed42_66e5790e at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0/runs/980e341d031346a2a1ddcd1af04610aa
🧪 View experiment at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0
Run: hind_Fifth_MLP_128-64_validation_seed42_66e5790e
Run ID: 980e341d031346a2a1ddcd1af04610aa


,model,trial,accuracy,precision,recall,f1,loss,AUC,fit_seconds,hyperparams,run_id
0,MLP,128-64,0.996392,0.996289,0.995711,0.995968,0.023219,0.999914,62.997364,"{'hidden_layer_sizes': (128, 64), 'learning_ra...",980e341d031346a2a1ddcd1af04610aa


In [ ]:
logreg_model = run_trial("LOGREG",
    trial_name="C0.1", C=0.1, max_iter=1000)

🏃 View run hind_Fifth_LOGREG_c0.1_validation_seed42_6726e790 at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0/runs/7a45a1857f6e4f58adf78f1db6747ccc
🧪 View experiment at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0
Run: hind_Fifth_LOGREG_c0.1_validation_seed42_6726e790
Run ID: 7a45a1857f6e4f58adf78f1db6747ccc


,model,trial,accuracy,precision,recall,f1,loss,AUC,fit_seconds,hyperparams,run_id
0,LOGREG,C0.1,0.997075,0.997433,0.996345,0.996772,0.013303,0.999955,7.53405,"{'C': 0.1, 'max_iter': 1000}",7a45a1857f6e4f58adf78f1db6747ccc


In [ ]:
et_model = run_trial(
    "ET",
    trial_name="200trees-depth20", n_estimators=200, max_depth=20)

🏃 View run hind_Fifth_ET_200trees-depth20_validation_seed42_04a3cf85 at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0/runs/8a5a529561a348e693d246d8f651a70d
🧪 View experiment at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0
Run: hind_Fifth_ET_200trees-depth20_validation_seed42_04a3cf85
Run ID: 8a5a529561a348e693d246d8f651a70d


,model,trial,accuracy,precision,recall,f1,loss,AUC,fit_seconds,hyperparams,run_id
0,ET,200trees-depth20,0.992882,0.993865,0.982638,0.986306,0.488674,0.999921,4.517715,"{'n_estimators': 200, 'max_depth': 20}",8a5a529561a348e693d246d8f651a70d


In [30]:
dummy_model = run_trial("DUMMY", trial_name="prior")

🏃 View run hind_Fifth_DUMMY_prior_validation_seed42_0878b5ae at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0/runs/3b111dea498f4b08a2b06e0f04d04226
🧪 View experiment at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0
Run: hind_Fifth_DUMMY_prior_validation_seed42_0878b5ae
Run ID: 3b111dea498f4b08a2b06e0f04d04226


,model,trial,accuracy,precision,recall,f1,loss,AUC,fit_seconds,hyperparams,run_id
0,DUMMY,prior,0.062793,0.001281,0.020408,0.002412,3.730903,0.5,2.354234,{},3b111dea498f4b08a2b06e0f04d04226


In [31]:
leaderboard = (
    pd.DataFrame(results)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

display(leaderboard)

,model,trial,accuracy,precision,recall,f1,loss,AUC,fit_seconds,hyperparams,run_id
0,LOGREG,C0.1,0.997075,0.997433,0.996345,0.996772,0.013303,0.999955,7.534050,"{'C': 0.1, 'max_iter': 1000}",7a45a1857f6e4f58adf78f1db6747ccc
1,XGB,gpu-depth6-100trees,0.996977,0.997163,0.996370,0.996662,0.012998,0.999960,19.952889,"{'n_estimators': 100, 'max_depth': 6, 'learnin...",bd0697f7dffc4ba2964d29080f5c994a
2,MLP,128-64,0.996392,0.996289,0.995711,0.995968,0.023219,0.999914,62.997364,"{'hidden_layer_sizes': (128, 64), 'learning_ra...",980e341d031346a2a1ddcd1af04610aa
3,ET,200trees-depth20,0.992882,0.993865,0.982638,0.986306,0.488674,0.999921,4.517715,"{'n_estimators': 200, 'max_depth': 20}",8a5a529561a348e693d246d8f651a70d
4,RF,200trees-depth15,0.986154,0.989226,0.957521,0.965452,0.776168,0.999826,3.408822,"{'n_estimators': 200, 'max_depth': 15, 'min_sa...",ea874eb639174ac087d4ee650f1b0064
5,DT,depth10,0.421509,0.347510,0.311847,0.308567,2.166304,0.787972,3.199208,"{'max_depth': 10, 'min_samples_leaf': 5}",0afe3f1c2422472c850c5a39ba519ef4
6,DUMMY,prior,0.062793,0.001281,0.020408,0.002412,3.730903,0.500000,2.354234,{},3b111dea498f4b08a2b06e0f04d04226
